# Pipeline Final

En este notebook trataremos como cargar el modelo fine-tuneado para inferir sobre un dataframe cualquiera con los mismos campos.

1. Importamos las librerias oportunas

In [1]:
import sys
import os
sys.path.append(os.path.abspath("../Code"))
import preproc
from transformers import pipeline
from sklearn.metrics import f1_score
import pandas as pd

2. Preprocesamos el dataframe sobre el que queremos inferir

In [ ]:
df = pd.read_csv("../Data/raw/tcga_simple_dev.csv")

print("Preprocesando...")
df["text"] = preproc.preprocesamiento(df["text"],3)

Preprocesando...


Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

[transformers] LongformerForSequenceClassification LOAD REPORT from: ../Models/longformer-prep_3_aug_len_1536
Key                  | Status     |  | 
---------------------+------------+--+-
loss_function.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Infiriendo...
F1-Score (macro): 0.8313


3. Cargamos el modelo e inferimos

In [ ]:
ruta_modelo_local = "../Models/longformer-prep_3_aug_len_1536"
nlp_pipeline = pipeline("text-classification", model=ruta_modelo_local, tokenizer=ruta_modelo_local)


textos = df['text'].tolist()
print("Infiriendo...")
resultados_pipeline = nlp_pipeline(textos, batch_size=8)

predicciones = [res['label'] for res in resultados_pipeline]

4. Mapeamos las etiquetas al formato requerido

In [ ]:
label_map = {'LABEL_0': 'T1', 'LABEL_1': 'T2', 'LABEL_2': 'T3', 'LABEL_3': 'T4'}
predicciones = [label_map[pred] for pred in predicciones]

etiquetas_reales = df['t'].tolist()

5. Obtenemos las métricas oportunas

In [ ]:
f1_macro = f1_score(etiquetas_reales, predicciones, average='macro')
print(f"F1-Score (macro): {f1_macro:.4f}")